# 01 — Data Preprocessing

## LLM Post-Training Experimental Study

Objective:
Prepare industrial maintenance datasets for
continued pretraining, supervised fine-tuning,
and direct preference optimization.

Pipeline:
PDF Extraction → Text Cleaning → Segmentation
→ Data Validation → Dataset Preparation

No model training is performed in this notebook.

In [1]:
%pip install -q pymupdf datasets pandas ##We use PyMuPDF for PDF extraction, HF datasets for daatset preparation, Pandas for inspection

In [2]:
from google.colab import drive

drive.mount('/content/drive') ##This allows the notebook to read the source data and save processed files persistently from Google Drive

Mounted at /content/drive


In [29]:
## Verifying the paths exist within Drive
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "llm-post-training-experimental-study"
)

RAW_PDF = (
    PROJECT_ROOT
    / "data/raw/industrial_maintenance_raw.pdf"
)

RAW_TXT = (
    PROJECT_ROOT
    / "data/raw/industrial_maintenance_raw_300_paragraphs.txt"
)

SFT_DATA = (
    PROJECT_ROOT
    / "data/instruction/"
    "industrial_maintenance_instruction_dataset.jsonl"
)

DPO_DATA = (
    PROJECT_ROOT
    / "data/preference/"
    "industrial_maintenance_dpo_preference_dataset.jsonl"
)

print("Raw PDF exists:", RAW_PDF.exists())
print("Raw Text exists:", RAW_TXT.exists())
print("SFT dataset exists:", SFT_DATA.exists())
print("DPO dataset exists:", DPO_DATA.exists())

Raw PDF exists: True
Raw Text exists: True
SFT dataset exists: True
DPO dataset exists: True


## Part 1: Non- instruction FT Dataset Preprocessing**

**Step 1: Extract Text from PDF**


In [4]:
import pymupdf

with pymupdf.open(RAW_PDF) as document:
    pages = [
        page.get_text("text")
        for page in document
    ]

raw_text = "\n".join(pages)

print("Pages extracted:", len(pages))
print("Characters extracted:", len(raw_text))

Pages extracted: 30
Characters extracted: 163179


**Step 2: Extract PDF page by page**

In [5]:
import pymupdf

def extract_pdf_pages(pdf_path):
    pages = []
    with pymupdf.open(pdf_path) as document:
        for number, page in enumerate(document, start=1):
            text = page.get_text('text')
            if text.strip():
                pages.append({'page': number, 'text': text})
    return pages

pdf_pages = extract_pdf_pages(RAW_PDF)
print('Extracted PDF pages:', len(pdf_pages))
print('First page (preview):', repr(pdf_pages[0]['text'][:350]))

Extracted PDF pages: 30
First page (preview): 'Page 1\nIndustrial Maintenance — Raw Text Dataset\n300 synthetic educational paragraphs for non-instruction fine-tuning. Not verified equipment instructions or manufacturer\ndocumentation.\nDuring a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in\nvibration near the drive-end bearing. The operator noted'


**Step 3: Clean PDF data**

Normalize Unicode, strip invisible characters, fix line-wrapped words, remove the dataset's repeating page headers, normalize spacing. We do not remove punctuation, equipment IDs, decimal points, units, or negations.

In [6]:
import re, unicodedata

def clean_pdf_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u200b', '').replace('\ufeff', '')
    text = re.sub(r'(?m)^Page\s+\d+\s*$', '', text)
    text = re.sub(r'(?m)^Industrial Maintenance\s*[—-]\s*Raw Text Dataset\s*$', '', text)
    text = re.sub(r'(?m)^300 synthetic educational paragraphs for non-instruction fine-tuning\..*$', '', text)
    text = re.sub(r'(?m)^documentation\.\s*$', '', text)  # wrapped header continuation in this PDF
    text = re.sub(r'(?<=\w)-\n(?=\w)', '', text)  # PDF hyphenated line wrap
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\s*\n\s*', ' ', text)  # rejoin wrapped visual lines
    return re.sub(r'\s+', ' ', text).strip()

cleaned_pages = [{'page': p['page'], 'text': clean_pdf_text(p['text'])} for p in pdf_pages]
print('Cleaned page preview:', cleaned_pages[0]['text'][:450])

Cleaned page preview: During a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in vibration near the drive-end bearing. The operator noted that the behavior was first noticed during the current shift. From an engineering perspective, the vibration trend may be associated with rotating assembly condition, coupling alignment, or support looseness. For traceability, recorded vibration direction, operating point, and inspect


**Step 4: Using Raw Text instead of PDF because of newline issues**

In [30]:
raw_text = RAW_TXT.read_text(encoding="utf-8")

# Split using blank lines between paragraphs
paragraphs = re.split(r"\n\s*\n", raw_text)

# Remove empty entries and unnecessary whitespace
paragraphs = [
    re.sub(r"\s+", " ", p).strip()
    for p in paragraphs
    if p.strip()
]

raw_dataset = pd.DataFrame({"text": paragraphs})

print("Number of records:", len(raw_dataset))
print("First record:", raw_dataset.iloc[0]["text"][:300])

Number of records: 1
First record: During a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in vibration near the drive-end bearing. The operator noted that the behavior was first noticed during the current shift. From an engineering perspective, the vibration trend may be associated wi


In [32]:
from datasets import Dataset

MIN_CHARS = 80
if RAW_TXT.exists():
    source_text = RAW_TXT.read_text(encoding='utf-8')
    candidates = [line.strip() for line in source_text.splitlines() if line.strip()]
    origin = 'original_txt_paragraphs_verified_against_pdf_source'
else:
    candidates = [p['text'] for p in cleaned_pages]
    origin = 'pdf_page_chunks_paragraph_boundaries_not_recoverable'

seen = set()
raw_records = []
for i, paragraph in enumerate(candidates, 1):
    paragraph = re.sub(r'\s+', ' ', paragraph).strip()
    if len(paragraph) < MIN_CHARS:
        continue
    key = paragraph.casefold()
    if key in seen:
        continue
    seen.add(key)
    raw_records.append({'text': paragraph, 'record_id': i, 'source_type': origin})

if not raw_records:
    raise ValueError('No usable raw-text records.')
raw_dataset = Dataset.from_list(raw_records)
print('Source:', origin)
print('Usable raw records:', len(raw_dataset))
print('Characters, first record:', len(raw_dataset[0]['text']))
print('Example:', raw_dataset[0]['text'][:400])

Source: original_txt_paragraphs_verified_against_pdf_source
Usable raw records: 300
Characters, first record: 607
Example: During a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in vibration near the drive-end bearing. The operator noted that the behavior was first noticed during the current shift. From an engineering perspective, the vibration trend may be associated with rotating assembly condition, coupling alignment, or support looseness. For traceability, recorded


In [39]:
print(type(raw_dataset))

<class 'datasets.arrow_dataset.Dataset'>


**Checking the data and sacing it in datasets path in drive**

In [40]:
import pandas as pd
from datasets import Dataset

# Convert to pandas if necessary
if isinstance(raw_dataset, Dataset):
    df = raw_dataset.to_pandas()
else:
    df = raw_dataset.copy()

# Validate the dataset
assert len(df) == 300
assert "text" in df.columns
assert df["text"].notna().all()
assert df["text"].str.strip().ne("").all()

# Check duplicates
print("Duplicate paragraphs:", df["text"].duplicated().sum())

# Save as JSONL
OUT.mkdir(parents=True, exist_ok=True)

output_file = OUT / "continued_pretraining.jsonl"

df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

# Reload to verify
saved_dataset = pd.read_json(output_file, lines=True)

assert len(saved_dataset) == len(df)

print("Dataset saved successfully!")
print("Location:", output_file)
print("Total records:", len(saved_dataset))
print("Total characters:", saved_dataset["text"].str.len().sum())

Duplicate paragraphs: 0
Dataset saved successfully!
Location: /content/drive/MyDrive/llm-post-training-experimental-study/data/processed/continued_pretraining.jsonl
Total records: 300
Total characters: 162440
